In [19]:
import MetaTrader5 as mt5
import pandas as pd
import time
import pytz
from datetime import datetime
import numpy as np
import requests
TOKEN = "7227666723:AAEsumQ2gWyr582xK3kGDwMFej0IvX1wD0s"
chat_id = "220684438"

mt5.initialize()


def get_values(symbol):
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_M30, 0, 200)
#     rates_frame = pd.DataFrame(rates)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
    rates_frame['rsi1'] = get_rsi(rates_frame['close'], 7)
    rates_frame['rsi2'] = get_rsi(rates_frame['close'], 14)

    # Calculate Supertren
    return rates_frame

def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

def Action_close(ticket_no, symbol, signal, lot):
    try:
        a = [[mt5.symbol_info_tick(symbol).ask, mt5.ORDER_TYPE_BUY], [mt5.symbol_info_tick(symbol).bid, mt5.ORDER_TYPE_SELL]]
        position_id=ticket_no
        price = a[signal][0]
        deviation=1000
        request={
            "action": mt5.TRADE_ACTION_DEAL,    
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][1],
            "position": position_id,
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script close",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result=mt5.order_send(request)
        return result
    except Exception as e:
        print("Action_close_Error")
        print(e)

def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)

def direction(a,j):
    if a.iloc[j].open < a.iloc[j].close:
        return 1
    else:
        return 0
    check = 0

def msg(message):
    url = f"https://api.telegram.org/bot{TOKEN}/sendMessage?chat_id={chat_id}&text={message}"
    r = requests.get(url)

def run(symbol):
    check = 0
    lot = 0.1
    buy_check = 0
    sell_check = 0
    buy_up = 0
    sell_up = 0
    order_time = 0
    old = 0
    old_pp = 0

    buy = 1
    sell = 0
 
    print(symbol)
    hour_passed = True

    while True:
        a = get_values(symbol)
        if a.iloc[-2].close != old:
            if a.iloc[-2].rsi1 <= 50 and a.iloc[-3].rsi1 <= 50 and a.iloc[-4].rsi1 <= 50 and a.iloc[-2].rsi1 >= 29 and \
                a.iloc[-3].rsi1 >= 30 and a.iloc[-4].rsi1 > 30 and a.iloc[-5].rsi1 >= 50 and sell_check == 0:
                if a.iloc[-6].rsi1 >= 70 :
                    if (a.iloc[-5].open - a.iloc[-5].close) <0.250 and (a.iloc[-4].open - a.iloc[-4].close) <0.250:             
                        message = f"Sell USDJPY --> {a.iloc[-1].name}"
                        print(a.iloc[-2])
#                         print(f"close -- {a.iloc[-2].close} ## ema1--{a.iloc[-2].rsi1} ## ema2--{a.iloc[-2].rsi2} ## {a.iloc[-2].name}")
                        url = f"https://api.telegram.org/bot{TOKEN}/sendMessage?chat_id={chat_id}&text={message}"
                        r = requests.get(url)
                        print(r.json())
                        old = a.iloc[-2].close
                        sellbuy_price = a.iloc[-1].close
                        result_sell = Action(symbol, lot, sell)
                        print(f"Symbol-->{symbol} ||| Type-->Sell  ||| Ticket_No-->{result_sell.order}")
                        sell_check = 1
#                         buy_check = 0
                    else:
                        pass
                else:
                    message = f"Sell USDJPY --> {a.iloc[-1].name}"
                    print(a.iloc[-2])
#                     print(f"close -- {a.iloc[-2].close} ## ema1--{a.iloc[-2].rsi1} ## ema2--{a.iloc[-2].rsi2} ## {a.iloc[-2].name}")
                    url = f"https://api.telegram.org/bot{TOKEN}/sendMessage?chat_id={chat_id}&text={message}"
                    r = requests.get(url)
                    print(r.json())
                    old = a.iloc[-2].close
                    sellbuy_price = a.iloc[-1].close
                    result_sell = Action(symbol, lot, sell)
                    print(f"Symbol-->{symbol} ||| Type-->Sell  ||| Ticket_No-->{result_sell.order}")
                    sell_check = 1
#                     buy_check = 0
                
            elif a.iloc[-2].rsi1 >= 50 and a.iloc[-3].rsi1 >= 50 and a.iloc[-4].rsi1 >= 50 and a.iloc[-2].rsi1 <= 71 and \
            a.iloc[-3].rsi1 <= 70 and a.iloc[-4].rsi1 <70 and a.iloc[-5].rsi1 <= 50 and buy_check == 0:
                if a.iloc[-6].rsi1 <= 29 :
                    if (a.iloc[-5].close - a.iloc[-5].open) <0.250 and (a.iloc[-4].close - a.iloc[-4].open) <0.250:             
                        message = f"BUY USDJPY --> {a.iloc[-1].name}"
                        url = f"https://api.telegram.org/bot{TOKEN}/sendMessage?chat_id={chat_id}&text={message}"
                        r = requests.get(url)
                        print(r.json())
                        old = a.iloc[-2].close
                        buybuy_price = a.iloc[-1].close
                        result_buy = Action(symbol, lot, buy)
                        print(f"Symbol-->{symbol} ||| Type-->Buy ||| Ticket_No-->{result_buy.order} ||| result_comment-->{result_buy.comment}")
                        buy_check = 1
#                         sell_check = 0
                    else:
                        pass
                else:
                    message = f"BUY USDJPY --> {a.iloc[-1].name}"
                    url = f"https://api.telegram.org/bot{TOKEN}/sendMessage?chat_id={chat_id}&text={message}"
                    r = requests.get(url)
                    print(r.json())
                    old = a.iloc[-2].close
                    buybuy_price = a.iloc[-1].close
                    result_buy = Action(symbol, lot, buy)
                    print(f"Symbol-->{symbol} ||| Type-->Buy ||| Ticket_No-->{result_buy.order} ||| result_comment-->{result_buy.comment}")
                    buy_check = 1
#                     sell_check = 0
                    
        if sell_check==1:
            sell_price = a.iloc[-2].close
            try:
                pp = mt5.positions_get(ticket=result_sell.order)[0].profit
            except Exception as e:
                print(e)
                pp = 0.0
            if (sellbuy_price-sell_price) > 0.920:
                result_sell = Action_close(result_sell.order, symbol, sell, lot)   #Action_close

                if result_sell.comment == "Requote":
                    result_sell = Action_close(result_sell.order, symbol, sell, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment}")
                sell_check = 0
                message = f"SELL Close USDJPY --> {a.iloc[-1].name}!! profit -->{pp}"
                msg(message)
            if (sell_price-sellbuy_price) > 0.100:
                result_sell = Action_close(result_sell.order, symbol, sell, lot)   #Action_close

                if result_sell.comment == "Requote":
                    result_sell = Action_close(result_sell.order, symbol, sell, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment}")
                sell_check = 0
                message = f"SELL Close USDJPY --> {a.iloc[-1].name}!! profit -->{pp}"
                msg(message)
            elif a.iloc[-2].rsi1 > 85:
                result_sell = Action_close(result_sell.order, symbol, sell, lot)   #Action_close

                if result_sell.comment == "Requote":
                    result_sell = Action_close(result_sell.order, symbol, sell, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment}")
                sell_check = 0
                message = f"SELL Close USDJPY --> {a.iloc[-1].name}!! profit -->{pp}"
                msg(message)
            elif a.iloc[-2].rsi1 < 10:
                result_sell = Action_close(result_sell.order, symbol, sell, lot)   #Action_close

                if result_sell.comment == "Requote":
                    result_sell = Action_close(result_sell.order, symbol, sell, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment}")
                sell_check = 0
                message = f"SELL Close USDJPY --> {a.iloc[-1].name}!! profit -->{pp}"
                msg(message)

        if buy_check==1:
            try:
                pp = mt5.positions_get(ticket=result_buy.order)[0].profit
            except Exception as e:
                print(e)
                pp = 0.0
            sell_price = a.iloc[-2].close
            if (sell_price-buybuy_price) > 0.920:
                result_buy = Action_close(result_buy.order, symbol, buy, lot)     #Action_close

                if result_buy.comment == "Requote":
                    result_buy = Action_close(result_buy.order, symbol, buy, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment}")  
                buy_check = 0
                message = f"BUY Close USDJPY --> {a.iloc[-1].name}!! profit -->{pp}"
                msg(message)
            if (buybuy_price - sell_price) > 0.100:
                result_buy = Action_close(result_buy.order, symbol, buy, lot)     #Action_close

                if result_buy.comment == "Requote":
                    result_buy = Action_close(result_buy.order, symbol, buy, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment}")  
                buy_check = 0
                message = f"BUY Close USDJPY --> {a.iloc[-1].name}!! profit -->{pp}"
                msg(message)
            elif a.iloc[-2].rsi1 < 15:
                result_buy = Action_close(result_buy.order, symbol, buy, lot)     #Action_close

                if result_buy.comment == "Requote":
                    result_buy = Action_close(result_buy.order, symbol, buy, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment}")  
                buy_check = 0
                message = f"BUY Close USDJPY --> {a.iloc[-1].name}!! profit -->{pp}"
                msg(message)
            elif a.iloc[-2].rsi1 > 90:
                result_buy = Action_close(result_buy.order, symbol, buy, lot)     #Action_close

                if result_buy.comment == "Requote":
                    result_buy = Action_close(result_buy.order, symbol, buy, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment}")  
                buy_check = 0
                message = f"BUY Close USDJPY --> {a.iloc[-1].name}!! profit -->{pp}"
                msg(message)

for symbol in ['USDJPY']:
    run(symbol)
    

USDJPY
open     142.910000
high     142.924000
low      142.579000
close    142.601000
rsi1      35.214085
rsi2      44.514763
Name: 2024-09-19 21:00:00, dtype: float64
{'ok': True, 'result': {'message_id': 252, 'from': {'id': 7227666723, 'is_bot': True, 'first_name': 'signal', 'username': 'usdema_bot'}, 'chat': {'id': 220684438, 'first_name': 'Animesh', 'last_name': 'Verma', 'username': 'xicor', 'type': 'private'}, 'date': 1726770600, 'text': 'Sell USDJPY --> 2024-09-19 21:30:00'}}
Symbol-->USDJPY ||| Type-->Sell  ||| Ticket_No-->5239099795
Close  Symbol-->USDJPY ||| Type-->Sell ||| result_comment-->Request executed


KeyboardInterrupt: 